<a href="https://colab.research.google.com/github/ElMartinez31/Data_Science/blob/main/Udacity_PEFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Apply lightweight finetuning to a fundation Model

# Tasks to perform
# 1. Load a pre-trained model and evaluate its performance
# 2. Perform parameter-efficient fine tuning using the pre-trained model
# 3. Perform inference using the fine-tuned model and compare its performance to the original model

In [ ]:
# Training a model using Hugging Face PEFT requires two additional steps beyond traditional fine-tuning:

# Creating a PEFT config
# Converting the model into a PEFT model using the PEFT config
# Inference using a PEFT model is almost identical to inference using a non-PEFT model. The only difference is that it must be loaded as a PEFT model.

In [ ]:
pip install -U transformers datasets peft accelerate evaluate


In [ ]:
# ===============
# imports 
# ===============

import os, json
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import  get_linear_schedule_with_warmup, DataCollatorWithPadding

from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from torch.utils.data import TensorDataset, DataLoader
from transformers import AutoModelForSequenceClassification
from peft import PeftModel


In [ ]:
# =========================
# 0) CONFIG & PATHS
# =========================
ID2LABEL = {0: "BAD", 1: "NEU", 2: "GOOD"}
LABEL2ID = {"BAD": 0, "NEU": 1, "GOOD": 2}
ADAPTER_DIR = "models/gpt2_lora_adapters/"
RESULTS_DIR = "results"
os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)



In [ ]:
# =========================
# 1) DATASET
# =========================
ds = load_dataset("prasadsawant7/sentiment_analysis_preprocessed_dataset")
ds

In [ ]:
# =========================
# 2) TOKENIZER
# =========================
tokenizer = AutoTokenizer.from_pretrained("gpt2")
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
tokenizer.padding_side = "left"

In [ ]:
# =========================
# 3) SANITIZE + TOKENIZE
# =========================
def sanitize_data(batch):
    texts = batch["text"]
    texts = [" " if (t is None) else t for t in texts]
    return {"text" : texts}

ds = ds.map(sanitize_data, batched=True)

def tokenize_sentences(batch):
    token = tokenizer(batch["text"], truncation=True, max_length=128)  # no padding here
    return token

ds_token_train = ds["train"].map(tokenize_sentences, batched=True, batch_size=1000)
ds_token_train
ds_token_test  = ds["test"].map(tokenize_sentences,  batched=True, batch_size=1000)

In [ ]:

# =========================
# 4) SELECT COLUMNS & DATALOADERS
# =========================
columns_to_keep = ["labels", "input_ids", "attention_mask"]

ds_token_train_ready = ds_token_train.select_columns(columns_to_keep)
ds_token_test_ready  = ds_token_test.select_columns(columns_to_keep)

ds_token_train_torch = ds_token_train_ready.set_format(type="torch", columns=columns_to_keep)
ds_token_test_torch  = ds_token_test_ready.set_format(type="torch",  columns=columns_to_keep)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_loader = DataLoader(ds_token_train_ready, batch_size=16, shuffle=True,  collate_fn=data_collator)
test_loader  = DataLoader(ds_token_test_ready,  batch_size=16, shuffle=False, collate_fn=data_collator)

In [ ]:
# =========================
# 5) BUILD BASE GPT-2 HEAD (for LoRA backbone)
# =========================
model = AutoModelForSequenceClassification.from_pretrained(
    "gpt2",
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
model.resize_token_embeddings(len(tokenizer))  # added [PAD]
model.config.pad_token_id = tokenizer.pad_token_id

In [ ]:

# =========================
# 6) PEFT LoRA CONFIG + WRAP
# =========================
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["c_attn", "c_fc", "c_proj"],
    bias="none",
    modules_to_save=["score"],  # keep the classification head trainable/saved
)
model = get_peft_model(model, peft_config)  # LoRA wrapping (freezes backbone params)
model.print_trainable_parameters()


In [ ]:

# =========================
# 7) TRAIN LoRA MODEL (2 epochs)
# =========================
optim = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

epochs = 2
for epoch in range(epochs):
    model.train()
    steps, running_loss = 0, 0.0
    for step, batch in enumerate(train_loader, start=1):
        optim.zero_grad()
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)  # returns loss if labels present
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optim.step()
        running_loss += loss.item()
        if step % 100 == 0 or step == len(train_loader):
            print(f'[epoch {epoch}] step {step}/{len(train_loader)} loss={running_loss/step:.4f}')


In [ ]:

# =========================
# 8) SAVE LoRA ADAPTERS + TOKENIZER
# =========================
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

In [ ]:

# =========================
# 9) RELOAD LoRA FOR INFERENCE
# =========================
base = AutoModelForSequenceClassification.from_pretrained(
    "gpt2",
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
base.resize_token_embeddings(len(tokenizer))
base.config.pad_token_id = tokenizer.pad_token_id
base.to(device)

model = PeftModel.from_pretrained(base, ADAPTER_DIR).to(device)

In [ ]:
# =========================
# 10) EVAL FUNCTION
# =========================
@torch.no_grad()
def eval_loop(model, loader):
    model.eval()
    correct = 0
    total = 0
    running_loss = 0.0
    steps = 0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)  # loss available if 'labels' is present
        loss = outputs.loss
        logits = outputs.logits
        preds = logits.argmax(dim=-1)
        running_loss += loss.item()
        steps += 1
        labels = batch["labels"]
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    avg_loss = running_loss / max(steps, 1)
    accuracy = correct / max(total, 1)
    return {"accuracy": accuracy, "average_loss": avg_loss}


In [ ]:

# =========================
# 11) EVALUATE LoRA MODEL
# =========================
lora_metrics = eval_loop(model=model, loader=test_loader)
print("LoRA model accuracy:", lora_metrics["accuracy"])


In [ ]:
# =========================
# 12) EVALUATE BASELINE (fresh GPT-2 + new head)
# =========================
base = AutoModelForSequenceClassification.from_pretrained(
    "gpt2",
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
base.resize_token_embeddings(len(tokenizer))
base.config.pad_token_id = tokenizer.pad_token_id
base.to(device)

baseline_metrics = eval_loop(base, test_loader)
print("Baseline model accuracy:", baseline_metrics["accuracy"])


In [ ]:



# =========================
# 13) COMPARISON + SAVE METRICS TO results/
# =========================
print("\n=== Comparison (Accuracy / Avg Loss) ===")
print(f"Baseline -> acc: {baseline_metrics['accuracy']:.4f} | loss: {baseline_metrics['average_loss']:.4f}")
print(f"LoRA     -> acc: {lora_metrics['accuracy']:.4f} | loss: {lora_metrics['average_loss']:.4f}")

# Ensure pure Python floats for JSON
baseline_json = {k: float(v) for k, v in baseline_metrics.items()}
lora_json     = {k: float(v) for k, v in lora_metrics.items()}

with open(os.path.join(RESULTS_DIR, "baseline_metrics.json"), "w") as f:
    json.dump(baseline_json, f, indent=2)
with open(os.path.join(RESULTS_DIR, "lora_metrics.json"), "w") as f:
    json.dump(lora_json, f, indent=2)

print("\nSaved:")
print(" - results/baseline_metrics.json")
print(" - results/lora_metrics.json")
print(" - models/gpt2_lora_adapters/ (adapters + tokenizer)")
